# Lecture 37


In [ ]:
from datascience import *
import numpy as np
import matplotlib

%matplotlib inline
import matplotlib.pyplot as plt
plt.style.use('fivethirtyeight')

import warnings
warnings.simplefilter("ignore")

## Review

### The $k$-Nearest Neighbors algorithm

In [ ]:
def distance(point1, point2):
    """Returns the distance between point1 and point2
    where each argument is an array 
    consisting of the coordinates of the point"""
    return np.sum((point1-point2)**2) ** 0.5

In [ ]:
def all_distances(training, new_row):
    """Returns an array of distances
    between each point in the training set
    and the new point (which is a row of attributes)"""
    attributes = training.drop('Class')
    def distance_from_new(row):
        return distance(make_array(new_row), make_array(row))
    return attributes.apply(distance_from_new)

In [ ]:
def table_with_distances(training, new_point):
    """Augments the training table 
    with a column of distances from new_point"""
    return training.with_column('Distance', all_distances(training, new_point))

In [ ]:
def nearest(training, new_point, k):
    """Returns a table of the k rows of the augmented table
    corresponding to the k smallest distances"""
    with_dists = table_with_distances(training, new_point)
    sorted_by_distance = with_dists.sort('Distance')
    nearest_neighbors_table = sorted_by_distance.take(np.arange(k))
    return nearest_neighbors_table

In [ ]:
def majority(nearest_neighbors_table, class_name):
    return nearest_neighbors_table.group(class_name).sort('count', 
                                                         descending=True).column(class_name).item(0)

In [ ]:
def one_knn(training, class_name, new_point, k):
    nearest_neighbors_table = nearest(training, new_point, k)
    return majority(nearest_neighbors_table, class_name)

### `patients` dataset

In [ ]:
patients = Table().read_table('https://ucb-dsus-adopters.github.io/materials-fds-assets-v2/lectures/lec37/breast-cancer.csv')
patients = patients.select('Single Epithelial Cell Size','Bland Chromatin', 'Class')
patients.sample(5)

In [ ]:
patients.group('Class')

#### **Task**: Split the data into `train` and `test` sets of (roughly) equal size.




$$  $$ 
$$  $$ 
$$  $$ 
$$  $$ 
$$  $$ 



- **Discussion Question**: Where does standardization play a role here?







In [ ]:
halfway = round(patients.num_rows/2)
halfway

In [ ]:
shuffled = patients.sample(with_replacement=False)

In [ ]:
train = shuffled.take(np.arange(halfway))
test = shuffled.take(np.arange(halfway, patients.num_rows))

In [ ]:
def standard_units(x):
    return (x - np.average(x)) / np.std(x)

In [ ]:
train = train.select('Class').with_columns(
    'Single Epithelial Cell Size', standard_units(train.column('Single Epithelial Cell Size')),
    'Bland Chromatin', standard_units(train.column('Bland Chromatin')),
    
)

In [ ]:
test = test.select('Class').with_columns(
    'Single Epithelial Cell Size', standard_units(test.column('Single Epithelial Cell Size')),
    'Bland Chromatin', standard_units(test.column('Bland Chromatin')),
)

#### **Task**: Make a classification using 5-nearest neighbors for every point in the testing set!

In [ ]:
def knn(training, class_name, testing, k):
    """Returns the test predictions in the test set in a table, along with their actual classifications"""
    
    predictions = make_array()
    for i in np.arange(testing.num_rows):
        predictions = np.append(predictions, one_knn(training, 
                                                     class_name, 
                                                     testing.drop(class_name).row(i), 
                                                     k))

    knn_table = Table().with_columns('Actual', testing.column(class_name),
                                     'Predicted', predictions)
    
    return knn_table 

In [ ]:
results = knn(train, 'Class', test, 5)
results

#### **Task**: Evaluate the accuracy of the classifier using $k=5$.

$$\text{Misclassification Rate (MCR)} = \text{proportion of testing points that are misclassified}$$

#### **Task**: Look into more detail about how our classifier went wrong.

**Discussion Question**: How many rows and columns will the table produced by the following command have?

```
results.group(['Actual', 'Predicted'])
```


## Suprise!

### Which $k$ should we choose?

We can figure out which one based on the evaluation metric, $\text{MCR}$. 

This step should be performed on the training set, *not* the testing set. Usually, we end up splitting the original training set into two parts: 

- "new" training set
- validation set.

In [ ]:
seventy = round(train.num_rows*(7/10))
seventy

In [ ]:
shuffled_train = train.sample(with_replacement=False)

In [ ]:
new_train = shuffled_train.take(np.arange(seventy))
validation = shuffled_train.take(np.arange(seventy, train.num_rows))

In [ ]:
new_train.num_rows

In [ ]:
validation.num_rows

In [ ]:
def which_k(training, class_name, validation, k_values):

    mc_rates = make_array()

    for i in np.arange(np.size(k_values)):

        results = knn(training, class_name, validation, k_values.item(i))
        new_rate = np.average(results.column('Actual') != results.column('Predicted'))
        mc_rates = np.append(mc_rates, new_rate)

    k_table = Table().with_columns("k", k_values,
                                   "MCR", mc_rates)
                             
    return k_table

In [ ]:
k_s = make_array(1,3,5,7,9,11,13,15)

In [ ]:
k_table = which_k(new_train, 'Class', validation, k_s)

In [ ]:
k_table

**Discussion Question**: 

$$ $$
$$ $$
$$ $$
$$ $$

What plot type would work best to visualize how the $\text{MCR}$ increases and decreases as the number of neighbors increases?

In [ ]:
k_table.plot('k','MCR')
plt.title('The best k for the job is...?');

#### **Task**: Evaluate our classifier's performance on the test set using the best $k$.

In [ ]:
results = knn(train, 'Class', test, 3)
results

In [ ]:
np.average(results.column('Actual') != results.column('Predicted'))